# 数据科学大作业 · Colab 运行入口

本笔记本用于在 **Google Colab** 上运行本项目（因果语言建模 · MHA/GQA/MQA 对比实验）。

## 实验设计

三种注意力机制各选一个 1B 级代表模型，在 **wikitext-2** 上做因果语言建模对比：

| 注意力 | 模型 | 说明 |
|--------|------|------|
| MHA | `facebook/opt-1.3b` | 标准多头注意力 |
| GQA | `meta-llama/Llama-3.2-1B` | 分组查询注意力 |
| MQA | `tiiuae/falcon-rw-1b` | 多查询注意力 |

## 运行前准备

1. `修改 → 笔记本设置 → 硬件加速器` 选择 **T4** 或 **A100**。
2. 左侧 🔑 **Secrets** 里添加密钥 `HF_TOKEN`（需已申请 Llama-3.2-1B 访问权限）。
3. 依次执行下面每个 cell。

## 1. 检查 GPU 与基础环境

In [ ]:
!nvidia-smi

## 2. 挂载 Google Drive（结果持久化）

Colab 重启会清空本地磁盘，因此把 `results/` 软链到 Drive，跑完可随时回看。

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/ds_project_results'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print('Drive 结果目录：', DRIVE_ROOT)

## 3. 克隆 GitHub 仓库

In [ ]:
%cd /content
!rm -rf DS_project
!git clone --depth 1 https://github.com/Frederick2313072/DS_project.git
%cd /content/DS_project
!git log -1 --oneline

## 4. 安装依赖

In [ ]:
!pip install -q -r requirements.txt
# 预初始化中文字体，避免第一次画图时卡住
!python -c "from mplfonts.bin.cli import init; init()"

## 5. HuggingFace 登录（读取 Colab Secrets 的 HF_TOKEN）

In [ ]:
from google.colab import userdata
from huggingface_hub import login

hf_token = userdata.get('HF_TOKEN')
login(token=hf_token)
print('HuggingFace 登录成功')

## 6. 将 results/ 软链到 Drive

这样训练脚本写到 `./results` 的文件会直接落到 Drive。

In [ ]:
import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
!rm -rf /content/DS_project/results
!ln -s {DRIVE_ROOT} /content/DS_project/results
!ls -l /content/DS_project/results

## 7. 定义单实验运行函数

逐个 cell 调用 `run_one(model, dataset)`，每个实验跑完自动释放显存、追加结果到 `results/results_lm.csv`。

调整训练规模：修改下面 `BASE_KWARGS` 中的 `num_train_epochs` / `max_train_samples` 等字段。

In [ ]:
import os, gc, sys, time
import torch
import pandas as pd

sys.path.insert(0, os.path.abspath("src"))
from train_lm import LMExperimentArgs, run_experiment, MODEL_CONFIGS

BASE_KWARGS = dict(
    seed=42,
    max_seq_len=512,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,
    warmup_ratio=0.05,
    weight_decay=0.01,
    fp16=True,
    max_train_samples=None,    # 冒烟测试可改 500
    max_eval_samples=5000,
    output_dir="./results",
    results_csv="./results/results_lm.csv",
)

def run_one(model_name: str, dataset: str):
    """跑单个 (model, dataset) 组合；跑完追加 CSV、释放显存、打印累计进度"""
    t0 = time.time()
    bar = "=" * 60
    print(f"\n{bar}\n>>> 实验开始: {model_name} × {dataset}\n{bar}")
    kwargs = dict(BASE_KWARGS, model_name=model_name, dataset=dataset)
    args = LMExperimentArgs(**kwargs)
    run_experiment(args)

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    csv_path = BASE_KWARGS["results_csv"]
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        done = df[["model","dataset"]].drop_duplicates().values.tolist()
        elapsed = (time.time() - t0) / 60
        print(f"<<< 完成: {model_name} × {dataset}  耗时 {elapsed:.1f} min")
        print(f"已累计完成 {len(done)} 组实验: {done}")

print("run_one() 已就绪，可以逐个 cell 跑实验了")

## 8. 逐个实验运行

三个实验各对应一种注意力机制，统一在 **wikitext** 上对比。

- **按顺序点运行**，跑完一个看结果再跑下一个；
- **中途停下来**，结果已在 CSV 和 Drive 里，下次打开 notebook 直接从下一个 cell 继续；
- **跳过某个模型**（如没有 LLaMA-3 授权）：跳过对应 cell 即可。

### 8.1  opt-1.3b × wikitext（MHA）

In [ ]:
run_one("opt-1.3b", "wikitext")

### 8.2  llama3-1b × wikitext（GQA）

In [ ]:
run_one("llama3-1b", "wikitext")

### 8.3  falcon-1b × wikitext（MQA）

In [ ]:
run_one("falcon-1b", "wikitext")

## 9. 查看 CSV 结果

In [ ]:
import pandas as pd
df = pd.read_csv('results/results_lm.csv')
df

## 10. 生成可视化图表

In [ ]:
!python src/visualize_lm.py

In [ ]:
# 把 results/ 下所有 png 图嵌入显示
from IPython.display import Image, display
import glob
for p in sorted(glob.glob('results/**/*.png', recursive=True)):
    print(p)
    display(Image(p))

## 11.（可选）打包下载结果

Drive 已自动同步，如需本地备份可执行下面这段。

In [ ]:
from google.colab import files
!tar -czf ds_results.tar.gz -C results .
files.download('ds_results.tar.gz')